In [11]:
import requests
import xml.etree.ElementTree as ET
import csv
import time

In [12]:
def get_main_mainArticles():
    url = 'https://edition.cnn.com/sitemap/article.xml'
    response = requests.get(url)
    if response.status_code != 200:
        print('Failed to get articles')
        return None
    root = ET.fromstring(response.content)
    mainArticles = []
    for child in root:
        mainArticles.append({
            'article_url': child[0].text,
            'last_updated': child[1].text
        })

    return mainArticles

In [13]:
def get_articles(mainArticle):
    response = requests.get(mainArticle['article_url'])
    if response.status_code != 200:
        print('Failed to get article')
        return []
    root = ET.fromstring(response.content)

    namespaces = {
        '': "http://www.sitemaps.org/schemas/sitemap/0.9",
        'image': "http://www.google.com/schemas/sitemap-image/1.1",
        'xhtml': "http://www.w3.org/1999/xhtml"
    }


    urls = root.findall(f'.//url', namespaces)
    article = []
    for url in urls:
        
        image_loc = url.findall(".//image:image/image:loc", namespaces)
        image_caption = url.findall(".//image:image/image:caption", namespaces)
        #  <xhtml:link rel="alternate" hreflang="x-default" href="https://edition.cnn.com/2024/10/19/us/wanda-dench-breast-cancer-thanksgiving-grandma/index.html"/>
        link_alternate = url.findall(".//xhtml:link[@rel='alternate']", namespaces)
        article.append({
            'loc': url.find('loc', namespaces).text,
            'lastmod': url.find('lastmod', namespaces).text,
            'image_loc': image_loc[0].text if image_loc else None,
            'image_caption': image_caption[0].text if image_caption else None,
            'link_alternate': link_alternate[0].attrib['href'] if link_alternate else None
        })
    return article

In [14]:
timeStart = time.time()

mainArticlesLists = get_main_mainArticles()

csv_file = 'articles.csv'
csv_columns = ['loc', 'lastmod', 'image_loc', 'image_caption', 'link_alternate']
with open(csv_file, mode='w', newline='', encoding='utf-8') as file:
    writer = csv.DictWriter(file, fieldnames=csv_columns)
    writer.writeheader()

    for mainArticlesList in mainArticlesLists:

        # display progress
        print(f"Getting articles from {mainArticlesList['article_url']}")
        progress = (mainArticlesLists.index(mainArticlesList) + 1) / len(mainArticlesLists) * 100
        print(f"Progress: {mainArticlesLists.index(mainArticlesList) + 1}/{len(mainArticlesLists)}", f"percent complete: {progress:.2f}%")

        # get articles
        articles = get_articles(mainArticlesList)
        

        # write articles to csv
        print(f"Writing {len(articles)} articles to {csv_file}")
        for article in articles:
            writer.writerow(article)



# display time taken
timeEnd = time.time()
print(f"Time taken: {timeEnd - timeStart} seconds")




Getting articles from https://www.cnn.com/sitemap/article/entertainment/2024/10.xml
Progress: 1/1413 percent complete: 0.07%
Writing 119 articles to articles.csv
Getting articles from https://www.cnn.com/sitemap/article/business/2024/10.xml
Progress: 2/1413 percent complete: 0.14%
Writing 259 articles to articles.csv
Getting articles from https://www.cnn.com/sitemap/article/world/2024/10.xml
Progress: 3/1413 percent complete: 0.21%
Writing 218 articles to articles.csv
Getting articles from https://www.cnn.com/sitemap/article/sport/2024/10.xml
Progress: 4/1413 percent complete: 0.28%
Writing 197 articles to articles.csv
Getting articles from https://www.cnn.com/sitemap/article/us/2024/10.xml
Progress: 5/1413 percent complete: 0.35%
Writing 205 articles to articles.csv
Getting articles from https://www.cnn.com/sitemap/article/politics/2024/10.xml
Progress: 6/1413 percent complete: 0.42%
Writing 435 articles to articles.csv
Getting articles from https://www.cnn.com/sitemap/article/health/

KeyboardInterrupt: 

In [ ]:
# print count rows in csv file
with open(csv_file, mode='r', encoding='utf-8') as file:
    reader = csv.reader(file)
    print(f"Total articles written to {csv_file}: {len(list(reader)) - 1}")